In [1]:
from explainability.models import LLMConfig
from explainability.models import DefaultHFModelLoader, BnBQuantizedModelLoader
from explainability.integrated_gradients import IGGenerate
from explainability.integrated_gradients import logprob_generate_forward_func

/datassd/home/plazaro/.conda/envs/qwen-unsloth-tunedlens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# First example with no quantization
model_config = LLMConfig(
                model_name="Qwen/Qwen2.5-3B",
                device_map={"": 1},
                torch_dtype="auto",
            )
model_loader = DefaultHFModelLoader(model_config)   # This will prompt for HF token
model, tokenizer = model_loader.load_model()
ig = IGGenerate(
    model=model,
    tokenizer=tokenizer,
    forward_func=lambda a,b,c,d: logprob_generate_forward_func(a,b,c,d, max_new_tokens = 20),
)
ig.compute_attributions(
    input_text="The quick brown fox jumps over the lazy dog",
    n_steps=30,
    integral_approximation_method="gausslegendre",
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.21s/it]


The => -1.5999
Ġquick => -0.3169
Ġbrown => -0.0335
Ġfox => 0.1023
Ġjumps => -0.0541
Ġover => -0.0029
Ġthe => 0.0288
Ġlazy => 0.0933
Ġdog => -0.0889


In [3]:
model_loader.prompt_generate(
    input_text="The quick brown fox jumps over the lazy dog",
    max_new_tokens=20,
)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'The quick brown fox jumps over the lazy dog.\nThe quick brown fox jumps over the lazy dog.'

In [3]:
ig.compute_attributions(
    input_text="The quick brown fox jumps over the lazy dog",
    n_steps=30,
    integral_approximation_method="gausslegendre",
)

The => -1.5999
Ġquick => -0.3169
Ġbrown => -0.0335
Ġfox => 0.1023
Ġjumps => -0.0541
Ġover => -0.0029
Ġthe => 0.0288
Ġlazy => 0.0933
Ġdog => -0.0889


In [ ]:
# Falta meter que me saque el output también, además del input con las attributions
# También tokenwise (?)
# También investigar qué pasa con el batch que me carga más en memoria
# Creo que tiene que ver con el n_steps, el captum está haciendo algo raro para meterlas todas de vez
# Creo que tengo que usar internal_batch_size 
# Considera también multiply_by_inputs 
# No debe ser inmediato, porque fijate en el numero de steps y tal, no lo se.

In [ ]:
# Para optimizar la VRAM, si no puedo mirar a ver qué n_steps es bueno basándome en la delta
# que es el error de la integral (no sé si de aproximar la integral o de los caminos)